In [1]:
%load_ext autoreload
%autoreload 2
from utils.plotting import *

In [2]:
# dir_path_seeds = np.array([
#     [
# "20260327-051120_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-050238_sup_seedpen_model_990",
# "20260327-052711_MLP_FSNet_seed3_nepochs300_lr0.0001_trainsize7000_finetune_20260327-050238_sup_seedpen_model_990",
# "20260327-052720_MLP_FSNet_seed1_nepochs300_lr0.0001_trainsize7000_finetune_20260327-050238_sup_seedpen_model_990",
# "20260327-051121_MLP_FSNet_seed2_nepochs300_lr0.0001_trainsize7000_finetune_20260327-050238_sup_seedpen_model_990",
#     ]
# ])

In [3]:
dir_path_seeds = np.array([
    [
        "20260327-095647_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_50",
    ],
    [
        "20260505-092325_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_150",
    ],
    [
        "20260327-120154_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_250",
    ],
    [
        "20260505-092359_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_350",
    ],
    [
        "20260505-092612_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_450",
    ],
    [
        "20260505-140412_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_550",
    ],
    [
        "20260505-141713_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_650",
    ],
    [
        "20260505-143021_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_750",
    ],
    [
        "20260505-144346_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_850",
    ],
    [
        "20260505-145630_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_950",
    ],
])

In [4]:
# dir_path_seeds = np.array([
#     [
#         "20260505-140135_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_50",
#     ],
#     [
#         "20260505-141430_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_150",
#     ],
#     [
#         "20260505-142752_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_250",
#     ],
#     [
#         "20260327-120154_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_250",
#     ],
#     [
#         "20260505-144120_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_350",
#     ],
#     [
#         "20260505-145418_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_450",
#     ],
#     [
#         "20260505-140052_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_550",
#     ],
#     [
#         "20260505-141309_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_650",
#     ],
#     [
#         "20260505-142519_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_750",
#     ],
#     [
#         "20260505-143621_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_850",
#     ],
#     [
#         "20260505-144809_MLP_FSNet_seed0_nepochs300_lr8e-05_trainsize7000_finetune_20260327-094852_sup_seedpen_model_950",
#     ],
# ])

In [5]:
# getting results
import os
import pickle
import yaml

rel_path = "./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000"

# collecting data
num_baselines = len(dir_path_seeds)
num_seeds_per_baseline = [len(seed_group) for seed_group in dir_path_seeds]
max_num_seeds = max(num_seeds_per_baseline)

obj_mean = np.full((num_baselines, max_num_seeds), np.nan, dtype=float)
obj_max = np.full((num_baselines, max_num_seeds), np.nan, dtype=float)
eq_violation_l1_mean = np.full((num_baselines, max_num_seeds), np.nan, dtype=float)
eq_violation_l1_max = np.full((num_baselines, max_num_seeds), np.nan, dtype=float)
ineq_violation_l1_mean = np.full((num_baselines, max_num_seeds), np.nan, dtype=float)
ineq_violation_l1_max = np.full((num_baselines, max_num_seeds), np.nan, dtype=float)
test_merit = np.full((num_baselines, max_num_seeds), np.nan, dtype=float)
model_size_mparams = np.full((num_baselines, max_num_seeds), np.nan, dtype=float)


def load_yaml_summary(path):
    """Load YAML summary with fallback for legacy python-tagged YAML files."""
    with open(path, "r") as f:
        text = f.read()

    try:
        return yaml.safe_load(text) or {}
    except yaml.constructor.ConstructorError:
        # Legacy summaries may contain Python object tags (e.g., TorchVersion).
        # These files are locally generated experiment artifacts.
        return yaml.unsafe_load(text) or {}


def load_test_metrics(dir_path, batch_size):
    """Load aggregated test metrics from new summary layout with old-layout fallback."""
    summary_path = os.path.join(dir_path, "test_summary.yaml")
    if os.path.exists(summary_path):
        summary = load_yaml_summary(summary_path)
        test_block = summary.get("test", {})

        # YAML may load numeric keys as int, old dumps may keep strings.
        bs_key = int(batch_size)
        result = test_block.get(bs_key, test_block.get(str(bs_key), None))
        if result is None:
            raise KeyError(f"batch size {batch_size} not found in {summary_path}")
        if "error" in result:
            raise RuntimeError(f"batch size {batch_size} has error: {result['error']}")
        return result

    # Backward compatibility: old results.pkl layout
    old_path = os.path.join(dir_path, "results.pkl")
    with open(old_path, "rb") as f:
        results = pickle.load(f)
    return results["test_results"]["batch_size_comparison"][batch_size]["metrics"]


def load_model_size_mparams(dir_path):
    """Load model size in millions of parameters from summary, with results.pkl fallback."""
    summary_path = os.path.join(dir_path, "test_summary.yaml")
    if os.path.exists(summary_path):
        summary = load_yaml_summary(summary_path)
        model_size = summary.get("model_size", {})
        if "total_params" in model_size:
            return float(model_size["total_params"]) / 1e6

    with open(os.path.join(dir_path, "results.pkl"), "rb") as f:
        results = pickle.load(f)
    total_params = results.get("model_size", {}).get("total_params", np.nan)
    return float(total_params) / 1e6 if not np.isnan(total_params) else np.nan


batch_size = 256
for i, seed_group in enumerate(dir_path_seeds):  # over baselines
    for j, run_name in enumerate(seed_group):  # over seeds in this baseline
        dir_path = os.path.join(rel_path, run_name)
        print(dir_path)
        results_ = load_test_metrics(dir_path, batch_size)

        # opt_gap in saved files is already in percent
        obj_mean[i, j] = results_["opt_gap_mean"]
        obj_max[i, j] = results_["opt_gap_max"]
        eq_violation_l1_mean[i, j] = results_["eq_violation_l1_mean"]
        eq_violation_l1_max[i, j] = results_["eq_violation_l1_max"]
        ineq_violation_l1_mean[i, j] = results_["ineq_violation_l1_mean"]
        ineq_violation_l1_max[i, j] = results_["ineq_violation_l1_max"]
        test_merit[i, j] = results_["merit_mean"]
        model_size_mparams[i, j] = load_model_size_mparams(dir_path)

# assuming you already have: dir_path_seeds, rel_path
print(num_baselines, num_seeds_per_baseline)

# read one file to know number of epochs (still from detailed results.pkl)
with open(os.path.join(rel_path, dir_path_seeds[0][0], "results.pkl"), "rb") as f:
    results = pickle.load(f)
    num_epochs = len(results["val_history"])

# store [num_baselines, max_num_seeds, num_epochs]
obj_mean_epochs = np.full((num_baselines, max_num_seeds, num_epochs), np.nan)

for i, seed_group in enumerate(dir_path_seeds):  # over baselines
    for j, run_name in enumerate(seed_group):  # over seeds in this baseline
        dir_path = os.path.join(rel_path, run_name)
        with open(os.path.join(dir_path, "results.pkl"), "rb") as f:
            results = pickle.load(f)
        # Optional: populate if val_history items include opt_gap_mean
        for k, entry in enumerate(results.get("val_history", [])):
            if isinstance(entry, dict) and "opt_gap_mean" in entry:
                obj_mean_epochs[i, j, k] = entry["opt_gap_mean"]
            else:
                obj_mean_epochs[i, j, k] = np.nan

./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260327-095647_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_50
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260505-092325_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_150
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260327-120154_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_250
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260505-092359_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_350
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260505-092612_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-094852_sup_seedpen_model_450
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260505-140412_MLP_FSNet_seed0_nepochs300_lr0.0001_trains

In [6]:
# Define metrics and headers
metrics = [
    ("Test Merit", test_merit),
    ("Eq Vio Mean", eq_violation_l1_mean),
    ("Eq Vio Max", eq_violation_l1_max),
    ("Ineq Vio Mean", ineq_violation_l1_mean),
    ("Ineq Vio Max", ineq_violation_l1_max),
    ("Opt Gap Mean", obj_mean),
    ("Opt Gap Max", obj_max),
    ("Size (M)", model_size_mparams),
]

# Print Header
header = f"{'Idx':<2} | " + " | ".join([f"{name:<20}" for name, _ in metrics])
print(header)
print("-" * len(header))

# Print Rows (Method)
for i in range(num_baselines):
    row_str = f"{i:<4} | "
    for _, data in metrics:
        # Compute mean and std over available seeds only
        mu = np.nanmean(data[i])
        sigma = np.nanstd(data[i])
        # Format as scientific notation
        if data is obj_mean or data is obj_max or data is test_merit:
            row_str += f"{mu:.2f} ± {sigma:.2f}".ljust(18) + " | "
        elif data is model_size_mparams:
            row_str += f"{mu:.3f} ± {sigma:.3f}".ljust(18) + " | "
        else:
            row_str += f"{mu:.2e} ± {sigma:.2e}".ljust(18) + " | "
    print(row_str)

Idx | Test Merit           | Eq Vio Mean          | Eq Vio Max           | Ineq Vio Mean        | Ineq Vio Max         | Opt Gap Mean         | Opt Gap Max          | Size (M)            
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
0    | 57.74 ± 0.00       | 5.88e-05 ± 0.00e+00 | 1.45e-03 ± 0.00e+00 | 3.16e-07 ± 0.00e+00 | 3.07e-05 ± 0.00e+00 | -2.98 ± 0.00       | 14.51 ± 0.00       | nan ± nan          | 
1    | 26.79 ± 0.00       | 2.90e-05 ± 0.00e+00 | 1.85e-03 ± 0.00e+00 | 2.23e-07 ± 0.00e+00 | 2.19e-05 ± 0.00e+00 | -5.18 ± 0.00       | 3.57 ± 0.00        | nan ± nan          | 
2    | 15.12 ± 0.00       | 1.72e-05 ± 0.00e+00 | 2.05e-03 ± 0.00e+00 | 3.78e-07 ± 0.00e+00 | 8.42e-05 ± 0.00e+00 | -5.11 ± 0.00       | 4.02 ± 0.00        | nan ± nan          | 
3    | 29.59 ± 0.00       | 2.96e-05 ± 0.00e+00 | 1.67e-03 ± 0.00e+00 | 6.52e-07 ± 0

/tmp/ipykernel_22734/502289110.py:23: RuntimeWarning: Mean of empty slice
  mu = np.nanmean(data[i])
/home/khain/.conda/envs/ml4opt/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
